## End to end Deep Learning Project Using Simple RNN

In [6]:
import ssl

# Permanently bypass SSL verification for all urllib requests in this session
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

In [7]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,Dense

In [8]:
## Load the imdb dataset

# max_features=10000 ##vocabulary size
# (X_train,y_train),(X_test,y_test)=imdb.load_data(num_words=max_features)

# # Print the shape of the data
# print(f'Training data shape: {X_train.shape}, Training labels shape: {y_train.shape}')
# print(f'Testing data shape: {X_train.shape}, Testing labels shape: {y_test.shape}')

In [9]:
import os
import numpy as np

# Points directly to the imdb.npz file inside your current working directory
file_path = 'imdb.npz'

# Load the data locally
with np.load(file_path, allow_pickle=True) as f:
    X_train, y_train = f['x_train'], f['y_train']
    X_test, y_test = f['x_test'], f['y_test']

# Apply the max_features filter manually
max_features = 10000
X_train = [[word for word in seq if word < max_features] for seq in X_train]
X_test = [[word for word in seq if word < max_features] for seq in X_test]

# Convert back to numpy arrays of objects (matching original behavior)
X_train = np.array(X_train, dtype=object)
X_test = np.array(X_test, dtype=object)

# Print shapes to verify it worked perfectly
print(f'Training data shape: {X_train.shape}, Training labels shape: {y_train.shape}')
print(f'Testing data shape: {X_test.shape}, Testing labels shape: {y_test.shape}')

Training data shape: (25000,), Training labels shape: (25000,)
Testing data shape: (25000,), Testing labels shape: (25000,)


In [10]:
X_train[0],y_train[0]
# one hot representation

([309,
  6,
  3,
  1069,
  209,
  9,
  2175,
  30,
  1,
  169,
  55,
  14,
  46,
  82,
  5869,
  41,
  393,
  110,
  138,
  14,
  5359,
  58,
  4477,
  150,
  8,
  1,
  5032,
  5948,
  482,
  69,
  5,
  261,
  12,
  2003,
  6,
  73,
  2436,
  5,
  632,
  71,
  6,
  5359,
  1,
  5,
  2004,
  1,
  5941,
  1534,
  34,
  67,
  64,
  205,
  140,
  65,
  1232,
  1,
  4,
  1,
  223,
  901,
  29,
  3024,
  69,
  4,
  1,
  5863,
  10,
  694,
  2,
  65,
  1534,
  51,
  10,
  216,
  1,
  387,
  8,
  60,
  3,
  1472,
  3724,
  802,
  5,
  3521,
  177,
  1,
  393,
  10,
  1238,
  30,
  309,
  3,
  353,
  344,
  2989,
  143,
  130,
  5,
  7804,
  28,
  4,
  126,
  5359,
  1472,
  2375,
  5,
  309,
  10,
  532,
  12,
  108,
  1470,
  4,
  58,
  556,
  101,
  12,
  309,
  6,
  227,
  4187,
  48,
  3,
  2237,
  12,
  9,
  215],
 np.int64(1))

In [11]:
## Inspect a sample review and its label
sample_review=X_train[0]
sample_label=y_train[0]

print(f"Sample review (as integers):{sample_review}")
print(f'Sample label: {sample_label}')


Sample review (as integers):[309, 6, 3, 1069, 209, 9, 2175, 30, 1, 169, 55, 14, 46, 82, 5869, 41, 393, 110, 138, 14, 5359, 58, 4477, 150, 8, 1, 5032, 5948, 482, 69, 5, 261, 12, 2003, 6, 73, 2436, 5, 632, 71, 6, 5359, 1, 5, 2004, 1, 5941, 1534, 34, 67, 64, 205, 140, 65, 1232, 1, 4, 1, 223, 901, 29, 3024, 69, 4, 1, 5863, 10, 694, 2, 65, 1534, 51, 10, 216, 1, 387, 8, 60, 3, 1472, 3724, 802, 5, 3521, 177, 1, 393, 10, 1238, 30, 309, 3, 353, 344, 2989, 143, 130, 5, 7804, 28, 4, 126, 5359, 1472, 2375, 5, 309, 10, 532, 12, 108, 1470, 4, 58, 556, 101, 12, 309, 6, 227, 4187, 48, 3, 2237, 12, 9, 215]
Sample label: 1


In [12]:
### MApping of words index back to words(for understanding)
word_index=imdb.get_word_index()
#word_index
reverse_word_index = {value: key for key, value in word_index.items()}
reverse_word_index

Exception: URL fetch failure on https://storage.googleapis.com/tensorflow/tf-keras-datasets/imdb_word_index.json: None -- [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1018)

In [ ]:
decoded_review = ' '.join([reverse_word_index.get(i - 3, '?') for i in sample_review])
decoded_review 

"? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what they have done don't you th

In [13]:
from tensorflow.keras.preprocessing import sequence

max_len=500

X_train=sequence.pad_sequences(X_train,maxlen=max_len)
X_test = sequence.pad_sequences(X_test, maxlen=max_len)
X_train

array([[   0,    0,    0, ...,   19,  178,   32],
       [   0,    0,    0, ...,   16,  145,   95],
       [   0,    0,    0, ...,    7,  129,  113],
       ...,
       [   0,    0,    0, ...,    4, 3586,    2],
       [   0,    0,    0, ...,   12,    9,   23],
       [   0,    0,    0, ...,  204,  131,    9]])

In [14]:
X_train[0]

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,   

In [22]:
## Train Simple RNN
model=Sequential()
model.add(Embedding(max_features,128,input_length=max_len)) ## Embedding Layers
model.add(SimpleRNN(128,activation='relu'))
model.add(Dense(1,activation="sigmoid"))

In [23]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 500, 128)          1280000   
                                                                 
 simple_rnn_1 (SimpleRNN)    (None, 128)               32896     
                                                                 
 dense_1 (Dense)             (None, 1)                 129       
                                                                 
Total params: 1313025 (5.01 MB)
Trainable params: 1313025 (5.01 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [24]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [25]:
## Create an instance of EarlyStoppping Callback
from tensorflow.keras.callbacks import EarlyStopping
earlystopping=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)
earlystopping

In [26]:
## Train the model with early sstopping
history=model.fit(
    X_train,y_train,epochs=10,batch_size=32,
    validation_split=0.2,
    callbacks=[earlystopping]
)

Epoch 1/10
625/625 [==============================] - 49s 77ms/step - loss: 9197.4336 - accuracy: 0.6139 - val_loss: 0.5950 - val_accuracy: 0.6720
Epoch 2/10
625/625 [==============================] - 48s 77ms/step - loss: 21616.6875 - accuracy: 0.6833 - val_loss: 0.6097 - val_accuracy: 0.6444
Epoch 3/10
625/625 [==============================] - 49s 78ms/step - loss: 0.5123 - accuracy: 0.7465 - val_loss: 0.5244 - val_accuracy: 0.7218
Epoch 4/10
625/625 [==============================] - 49s 79ms/step - loss: 16.1929 - accuracy: 0.7918 - val_loss: 0.6155 - val_accuracy: 0.6352
Epoch 5/10
625/625 [==============================] - 49s 78ms/step - loss: 52.9721 - accuracy: 0.7521 - val_loss: 0.6609 - val_accuracy: 0.5842
Epoch 6/10
625/625 [==============================] - 48s 77ms/step - loss: 0.5073 - accuracy: 0.7298 - val_loss: 0.5569 - val_accuracy: 0.7086
Epoch 7/10
625/625 [==============================] - 49s 79ms/step - loss: 0.3661 - accuracy: 0.8482 - val_loss: 0.5162 - val_

In [27]:
## Save model file
model.save('simple_rnn_imdb.h5')

e:\UDemy Final\Deep Learning NLP\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
